# Why Van Loan's Theorem Is Really Nice

**Theorem 1.17 (Van Loan).** Let **A**(x), **B**(x), and **C**(x) be continuous matrix functions. Then:

$$
\prod_s^t \left( I + \begin{pmatrix} A(x) & B(x) \\ 0 & C(x) \end{pmatrix} \mathrm{d}x \right)
= \begin{pmatrix} \prod_s^t (I + A(x)\,\mathrm{d}x) & \int_s^t \prod_s^u (I + A(x)\,\mathrm{d}x)\, B(u)\, \prod_u^t (I + C(x)\,\mathrm{d}x)\,\mathrm{d}u \\ 0 & \prod_s^t (I + C(x)\,\mathrm{d}x) \end{pmatrix}
$$

---

## The Big Idea

In life insurance, we constantly need to compute expressions of the form:

$$
\int_s^t \prod_s^u (I + A(x)\,\mathrm{d}x)\, B(u) \, \prod_u^t (I + C(x)\,\mathrm{d}x)\,\mathrm{d}u
$$

This integral looks **horrible** to compute directly:
- For each value of `u`, you need **two** product integrals: one from `s` to `u` and one from `u` to `t`
- Then you multiply all three matrices together
- Then you **integrate** over all `u` from `s` to `t`

Van Loan says: **don't do any of that**. Instead:
1. Glue A, B, C into a single block upper-triangular matrix
2. Compute **one** product integral of the big matrix
3. Read off the answer from the upper-right block

### Why this matters in practice

When intensities are **piecewise constant** (the standard assumption for Markov life insurance models), product integrals become **matrix exponentials** (by Thm 1.13 / Cor 1.14). So Van Loan reduces the problem to:

> Compute a single matrix exponential of a block matrix → read off the upper-right block.

This is what makes reserves, expected sojourn times, and higher-order moments computationally tractable.

## Numerical Example: A 3-State Disability Model

Consider a simple disability model with states:
- **0**: Active
- **1**: Disabled  
- **2**: Dead (absorbing)

The transition intensity matrix (for constant rates) is:

$$
\Lambda = \begin{pmatrix} -(\sigma + \mu_a) & \sigma & \mu_a \\ \rho & -(\rho + \mu_d) & \mu_d \\ 0 & 0 & 0 \end{pmatrix}
$$

**Goal**: Compute the state-wise expected sojourn time in state `j` given we start in state `i`:

$$
E_i\left[\int_0^T \mathbb{1}_{\{Z(u) = j\}}\,\mathrm{d}u\right] = \int_0^T P_{ij}(0, u)\,\mathrm{d}u
$$

This is exactly an integral of the form that Van Loan handles!

In [1]:
import numpy as np
from scipy.linalg import expm

# Transition intensities (constant)
sigma = 0.05   # active -> disabled
rho = 0.02     # disabled -> active (recovery)
mu_a = 0.01    # active -> dead
mu_d = 0.03    # disabled -> dead

# Intensity matrix (3x3)
Lambda = np.array([
    [-(sigma + mu_a),  sigma,          mu_a],
    [rho,              -(rho + mu_d),  mu_d],
    [0,                0,              0   ]
])

print("Intensity matrix Lambda:")
print(Lambda)
print(f"\nRow sums (should be 0): {Lambda.sum(axis=1)}")

Intensity matrix Lambda:
[[-0.06  0.05  0.01]
 [ 0.02 -0.05  0.03]
 [ 0.    0.    0.  ]]

Row sums (should be 0): [-1.73472348e-18 -3.46944695e-18  0.00000000e+00]


## Method 1: The Hard Way (Numerical Integration)

For constant intensities, P(0, u) = exp(Lambda * u). The expected sojourn time is:

$$
\int_0^T e^{\Lambda u}\,\mathrm{d}u
$$

We compute this by brute-force numerical quadrature: evaluate exp(Lambda * u) at many points and sum.

In [2]:
T = 10  # time horizon

# ----- Method 1: Brute-force numerical integration -----
n_steps = 10000
dt = T / n_steps
integral_bruteforce = np.zeros((3, 3))

for k in range(n_steps):
    u = k * dt
    integral_bruteforce += expm(Lambda * u) * dt  # trapezoidal-ish

print("Method 1 — Brute-force numerical integration:")
print(f"(using {n_steps} steps, each calling expm)")
print(np.round(integral_bruteforce, 6))

Method 1 — Brute-force numerical integration:
(using 10000 steps, each calling expm)
[[ 7.630201  1.761276  0.608523]
 [ 0.70451   7.982456  1.313034]
 [ 0.        0.       10.      ]]


## Method 2: Van Loan (One Matrix Exponential!)

We want to compute:

$$
\int_0^T e^{\Lambda u} \cdot I \cdot e^{0 \cdot (T-u)}\,\mathrm{d}u = \int_0^T e^{\Lambda u}\,\mathrm{d}u
$$

This fits Van Loan with **A** = Lambda, **B** = I, **C** = 0. So we build the block matrix:

$$
M = \begin{pmatrix} \Lambda & I \\ 0 & 0 \end{pmatrix}
$$

and compute exp(M * T). The **upper-right block** gives us the integral directly!

In [3]:
# ----- Method 2: Van Loan -----
n = Lambda.shape[0]  # 3

# Build the 6x6 block matrix
M = np.zeros((2*n, 2*n))
M[:n, :n] = Lambda    # upper-left: A = Lambda
M[:n, n:] = np.eye(n) # upper-right: B = I
# lower-left: 0 (already zero)
# lower-right: C = 0 (already zero)

print("Block matrix M (6x6):")
print(M)

# Compute ONE matrix exponential
result = expm(M * T)

# Read off the upper-right block
integral_vanloan = result[:n, n:]

print("\nMethod 2 — Van Loan (single expm call):")
print(np.round(integral_vanloan, 6))

Block matrix M (6x6):
[[-0.06  0.05  0.01  1.    0.    0.  ]
 [ 0.02 -0.05  0.03  0.    1.    0.  ]
 [ 0.    0.    0.    0.    0.    1.  ]
 [ 0.    0.    0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.    0.    0.  ]]

Method 2 — Van Loan (single expm call):
[[ 7.62999   1.761423  0.608588]
 [ 0.704569  7.982274  1.313157]
 [ 0.        0.       10.      ]]


In [4]:
# Compare the two methods
print("Difference between methods (should be ~0):")
print(np.round(integral_vanloan - integral_bruteforce, 8))
print(f"\nMax absolute error: {np.max(np.abs(integral_vanloan - integral_bruteforce)):.2e}")

Difference between methods (should be ~0):
[[-2.1129e-04  1.4672e-04  6.4570e-05]
 [ 5.8690e-05 -1.8194e-04  1.2326e-04]
 [ 0.0000e+00  0.0000e+00  0.0000e+00]]

Max absolute error: 2.11e-04


## Timing Comparison

Let's see just how much faster Van Loan is.

In [5]:
import time

# Time Method 1 (brute force)
start = time.perf_counter()
for _ in range(10):  # repeat for reliable timing
    integral_bf = np.zeros((3, 3))
    for k in range(n_steps):
        u = k * dt
        integral_bf += expm(Lambda * u) * dt
t_bruteforce = (time.perf_counter() - start) / 10

# Time Method 2 (Van Loan)
start = time.perf_counter()
for _ in range(10000):  # many repeats since it's so fast
    result = expm(M * T)
    integral_vl = result[:n, n:]
t_vanloan = (time.perf_counter() - start) / 10000

print(f"Brute force:  {t_bruteforce:.4f} seconds")
print(f"Van Loan:     {t_vanloan:.6f} seconds")
print(f"\nSpeedup: {t_bruteforce / t_vanloan:.0f}x faster!")

Brute force:  0.5004 seconds
Van Loan:     0.000069 seconds

Speedup: 7255x faster!


## A More Interesting Application: State-Weighted Reserves

In Liv2, we often need integrals like:

$$
\int_s^t P(s,u)\, \text{diag}(b)\, P(u,t)\, \mathrm{d}u
$$

where **b** is a vector of state-dependent benefit rates. This has the **sandwich** structure:
- Left product integral: P(s, u)
- Middle: B(u) = diag(b)
- Right product integral: P(u, t)

Without Van Loan, you'd need a double loop (over `u` and for each Runge-Kutta step inside P). With Van Loan, it's still **one** matrix exponential of:

$$
M = \begin{pmatrix} \Lambda & \text{diag}(b) \\ 0 & \Lambda \end{pmatrix}
$$

In [6]:
# Benefit rate: 1 unit per year while disabled (state 1)
b = np.array([0, 1, 0])  # benefit only in state 1
B = np.diag(b)

# Build the Van Loan block matrix: A = Lambda, B = diag(b), C = Lambda
M2 = np.zeros((2*n, 2*n))
M2[:n, :n] = Lambda   # upper-left: A = Lambda
M2[:n, n:] = B         # upper-right: B = diag(b)
M2[n:, n:] = Lambda   # lower-right: C = Lambda

result2 = expm(M2 * T)
sandwich_integral = result2[:n, n:]

print("Integral of P(0,u) * diag(b) * P(u,T) du from 0 to T:")
print(np.round(sandwich_integral, 6))
print()
print("Interpretation: entry (i,j) = expected discounted time in state 1")
print("while transitioning from state i to state j over [0, T].")

Integral of P(0,u) * diag(b) * P(u,T) du from 0 to T:
[[0.097148 1.491429 0.172847]
 [0.596571 6.264    1.121703]
 [0.       0.       0.      ]]

Interpretation: entry (i,j) = expected discounted time in state 1
while transitioning from state i to state j over [0, T].


In [7]:
# Verify against brute force
sandwich_bf = np.zeros((3, 3))
for k in range(n_steps):
    u = k * dt
    P_0u = expm(Lambda * u)
    P_uT = expm(Lambda * (T - u))
    sandwich_bf += P_0u @ B @ P_uT * dt

print("Brute-force result:")
print(np.round(sandwich_bf, 6))
print(f"\nMax error vs Van Loan: {np.max(np.abs(sandwich_integral - sandwich_bf)):.2e}")

Brute-force result:
[[0.097148 1.491282 0.172847]
 [0.59663  6.264    1.121826]
 [0.       0.       0.      ]]

Max error vs Van Loan: 1.47e-04


## Summary: Why Van Loan Is Really Nice

| Aspect | Without Van Loan | With Van Loan |
|--------|-----------------|---------------|
| **What you compute** | Loop over u: two product integrals + multiply + integrate | One product integral of a bigger matrix |
| **Piecewise constant** | Loop over u: two expm calls per step | **One** expm call total |
| **Speed** | O(N) matrix exponentials | O(1) matrix exponential |
| **Accuracy** | Depends on quadrature grid | Exact (up to machine precision) |
| **Code complexity** | Nested loops | 3 lines |

The price you pay: the matrix is bigger (2n × 2n instead of n × n). But for typical life insurance models with n = 3–10 states, this is negligible.

## Check Your Understanding

1. In the simple sojourn-time example, we used A = Lambda, B = I, C = 0. Explain *why* C = 0 makes sense (hint: what does the right product integral become when C = 0?).

2. If you wanted to compute the integral of exp(-r*u) * P(0,u) du (i.e., the *discounted* expected sojourn times with discount rate r), how would you modify the Van Loan block matrix? (Hint: think about Corollary 1.15.)

3. Van Loan requires the block matrix to be **upper triangular**. Why doesn't the theorem work if you put something nonzero in the lower-left block?